# Flood Control Projects — Stage 1: Load, Inspect & Clean
## Philippine Flood Control Projects Analysis

This notebook covers the first stage of the Python phase of the project:

1. **Load & structural inspection** — flattening the raw ArcGIS JSON export and checking its shape, missing values, and duplicates
2. **Cleaning** — handling repeated multi-year projects, converting dates, adding project duration, and filling missing municipalities

**Data source:** [BetterGov.PH Open Data Portal](https://data.bettergov.ph/) — Flood Control Projects Dashboard data (CC0 1.0 license, last updated Sep 21, 2025; downloaded [your download date]). One row per project component, with location, contractor, cost, and completion details.

The raw file contains 9,855 records covering infrastructure years 2018–2025. Exact row counts before and after cleaning are printed and verified in Sections 1 and 4 below.

**Output of this notebook:** one cleaned Excel file saved to `data/`, used as input for the EDA notebook that follows.

In [1]:
### Libraries
import json
import pandas as pd


## 1. Load

The raw file is an ArcGIS export, so the records are nested under
`features` → `attributes`. They are flattened into a single table here.

In [2]:
with open("../data/flood_control.json", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame([feature["attributes"] for feature in raw["features"]])
df_raw = df.copy() # untouched copy of the raw data for future reference

print(df.shape)

(9855, 33)


## 2. Inspect

Checking structure, missing values, and duplicates before making any changes.
The issues found here are handled in Section 3.

In [3]:
# column names, data types, and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9855 entries, 0 to 9854
Data columns (total 33 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   InfraYear                    9855 non-null   int64  
 1   Region                       9855 non-null   object 
 2   Province                     9855 non-null   object 
 3   Municipality                 9189 non-null   object 
 4   ImplementingOffice           9855 non-null   object 
 5   ProjectID                    9855 non-null   object 
 6   ProjectDescription           9855 non-null   object 
 7   ProjectComponentID           9855 non-null   object 
 8   ProjectComponentDescription  9855 non-null   object 
 9   Program                      0 non-null      object 
 10  TypeofWork                   9855 non-null   object 
 11  infra_type                   9855 non-null   object 
 12  Longitude                    9855 non-null   float64
 13  Latitude          

In [4]:
# first 5 rows of the dataframe
df.head(5)

,InfraYear,Region,Province,Municipality,ImplementingOffice,ProjectID,ProjectDescription,ProjectComponentID,ProjectComponentDescription,Program,...,EditDate,Editor,FundingYear,LegislativeDistrict,DistrictEngineeringOffice,GlobalID,ABC_String,ContractCost_String,CompletionDateActual,StartDate
0,2024,Region IV-B,PALAWAN,PUERTO PRINCESA CITY (CAPITAL) (PALAWAN),Palawan 3rd District Engineering Office,P00824759LZ,Construction of Babuyan River Flood Control St...,P00824759LZ-CW1,Construction of Flood Mitigation Structure - C...,None,...,1754303523936,dpwh_view,2024,PALAWAN (THIRD LEGISLATIVE DISTRICT),Palawan 3rd District Engineering Office,79e8be69-90ab-488d-bf10-9204d41338ab,17963475,17961569.07,2024-05-23,02/15/2024
1,2024,Region IV-B,PALAWAN,PUERTO PRINCESA CITY (CAPITAL) (PALAWAN),Palawan 3rd District Engineering Office,P00824758LZ,Construction of Flood Control Strucurue along ...,P00824758LZ-CW1,Construction of Flood Mitigation Structure - C...,None,...,1754303523936,dpwh_view,2024,PALAWAN (THIRD LEGISLATIVE DISTRICT),Palawan 3rd District Engineering Office,dea02c2f-d69d-4757-8d21-853e001736c5,17370000,17369992.99,2024-07-19,02/23/2024
2,2024,Region IV-B,PALAWAN,PUERTO PRINCESA CITY (CAPITAL) (PALAWAN),Palawan 3rd District Engineering Office,P00824750LZ,Construction of Babuyan River Flood Control St...,P00824750LZ-CW1,Construction of Flood Mitigation Structure - C...,None,...,1754303523936,dpwh_view,2024,PALAWAN (THIRD LEGISLATIVE DISTRICT),Palawan 3rd District Engineering Office,7177ab8b-b330-40f5-bdda-44ce7823ddf8,18335000,16802607.63,2024-07-24,02/23/2024
3,2023,Region IV-B,PALAWAN,PUERTO PRINCESA CITY (CAPITAL) (PALAWAN),Palawan 3rd District Engineering Office,P00737551LZ,"Construction of Flood Control Structure, Baran...",P00737551LZ-CW1,Construction of Drainage Structure - Construct...,None,...,1754303523936,dpwh_view,2023,PALAWAN (THIRD LEGISLATIVE DISTRICT),Palawan 3rd District Engineering Office,fcf681df-bf3b-47a6-8807-01f94106bef9,4900000,4899590,2023-05-05,02/15/2023
4,2024,Region IV-B,PALAWAN,PUERTO PRINCESA CITY (CAPITAL) (PALAWAN),Region IV-B,P00823565LZ,"Construction of Tanabag River Waterway, (Upstr...",P00823565LZ-CW1,Construction of Flood Mitigation Structure - C...,None,...,1754303523936,dpwh_view,2024,PALAWAN (THIRD LEGISLATIVE DISTRICT),Palawan 3rd District Engineering Office,a3d64d20-0d8d-4b61-abdd-38a3bd840c77,154399999.99,154395357.94,2024-09-05,03/09/2024


In [5]:
# missing values per column 
df.isnull().sum()

InfraYear                         0
Region                            0
Province                          0
Municipality                    666
ImplementingOffice                0
ProjectID                         0
ProjectDescription                0
ProjectComponentID                0
ProjectComponentDescription       0
Program                        9855
TypeofWork                        0
infra_type                        0
Longitude                         0
Latitude                          0
ContractID                        0
ABC                               0
ContractCost                      0
CompletionDateOriginal            4
CompletionYear                    0
Contractor                        0
ObjectId                          0
CreationDate                      0
Creator                           0
EditDate                          0
Editor                            0
FundingYear                       0
LegislativeDistrict               0
DistrictEngineeringOffice   

In [6]:
# number of fully duplicated rows in the dataframe
df.duplicated().sum()

0

In [7]:
# number of duplicated ProjectIDs in the dataframe
df["ProjectID"].duplicated().sum()

28

### Findings from inspection

- The dataset has 9,855 rows and 33 columns.
- `Program` is empty in every row, so it carries no information.
- `Municipality` is missing in 666 rows, and `CompletionDateOriginal` in 4.
- No rows are fully duplicated, but 28 `ProjectID` values repeat.
- `StartDate` and `CompletionDateActual` are stored as text, so they may hide
  empty or placeholder values that `isna()` does not catch.

### Repeated ProjectIDs

Although no rows are fully duplicated, 28 `ProjectID` values repeat. The repeated
rows are compared below to see which columns differ, which decides whether they
are true duplicates to drop or separate components to keep and flag.

In [8]:
# All rows that share a ProjectID with another row, grouped together
dup_ids = df[df["ProjectID"].duplicated(keep=False)].sort_values("ProjectID")
print(len(dup_ids), "rows share a repeated ProjectID")

# Key columns for a quick side-by-side comparison
cols = ["ProjectID", "ContractID", "Contractor", "ABC", "ContractCost",
        "Municipality", "Latitude", "Longitude", "ObjectId"]
dup_ids[cols].head(10)

55 rows share a repeated ProjectID


,ProjectID,ContractID,Contractor,ABC,ContractCost,Municipality,Latitude,Longitude,ObjectId
8960,P00320211VS,19I00115,NORTHERN BUILDERS / RIANNE CONSTRUCTION,233876034.9,227812286.4,None,11.278820,125.066788,8961
8955,P00320211VS,19I00115,NORTHERN BUILDERS / RIANNE CONSTRUCTION,233876034.9,227812286.4,None,11.278820,125.066788,8956
8951,P00320211VS,19I00115,NORTHERN BUILDERS / RIANNE CONSTRUCTION,233876034.9,227812286.4,None,11.278820,125.066788,8952
9629,P00420007MN,20LB0036,APEIRON CONSTRUCTION SOLUTIONS / ABU CONSTRUCTION,77523275.0,77523275.0,None,7.053756,125.606269,9631
9618,P00420007MN,20LB0036,APEIRON CONSTRUCTION SOLUTIONS / ABU CONSTRUCTION,77523275.0,77523275.0,None,7.053756,125.606269,9620
9098,P00420010MN,20LB0027,A.Y.U. ENGINEERING & CONSTRUCTION / ABU CONSTR...,86136865.0,86136865.0,None,7.060800,125.566770,9100
9099,P00420010MN,20LB0027,A.Y.U. ENGINEERING & CONSTRUCTION / ABU CONSTR...,86136865.0,86136865.0,None,7.060800,125.566770,9101
9534,P00420017MN,20LB0037,ABU CONSTRUCTION,60296095.0,60296095.0,None,7.065010,125.601380,9536
9526,P00420017MN,20LB0037,ABU CONSTRUCTION,60296095.0,60296095.0,None,7.065010,125.601380,9528
9539,P00420021MN,20LB0040,RAKKI CORPORATION / RELY CONSTRUCTION & SUPPLY...,86136865.0,86136865.0,None,7.065010,125.601380,9541


In [9]:
# Distinct values per column within each repeated ProjectID
distinct_counts = dup_ids.groupby("ProjectID").nunique()

# A column that never differs has a max of 1; keep only the columns that do differ
max_distinct = distinct_counts.max()
max_distinct[max_distinct > 1]

InfraYear              3
ObjectId               3
FundingYear            3
GlobalID               3
ABC_String             2
ContractCost_String    2
dtype: int64

### Finding: repeated ProjectIDs are multi-year contracts

The 28 repeated `ProjectID` values (27 distinct projects, 55 rows) are the same
contract listed once per funding year. Only `InfraYear`, `FundingYear`, and
identifier columns differ between the repeats. The string-formatted cost columns
of the later rows read "MYCA with Project ID ...", which points to a
multi-year contract agreement. The numeric `ContractCost` carries the full
contract amount in every row, so summing it without handling these rows would
count those projects two to three times.

## 3. Clean

All cleaning is done on a copy (`df_clean`), so the raw data in `df_raw` stays
untouched for comparison in Section 4.

### Decision: keep the first-year row for each multi-year project

Each repeated `ProjectID` is kept once, using the earliest `InfraYear` row, and
the later-year rows are dropped. The earliest row is the one holding the real
cost text in `ABC_String` and `ContractCost_String`, and keeping one row per
project means each contract's cost is counted once in totals. The trade-off is
that later funding years are no longer visible, which none of the analysis
questions need.

In [10]:
df_clean = df.copy()

rows_before = len(df_clean)

# Sort so the earliest year comes first, then keep one row per ProjectID
df_clean = (
    df_clean
    .sort_values(["ProjectID", "InfraYear"])
    .drop_duplicates(subset="ProjectID", keep="first")
    .reset_index(drop=True)
)

print(f"{rows_before} -> {len(df_clean)} rows ({rows_before - len(df_clean)} removed)")

9855 -> 9827 rows (28 removed)


In [11]:
# Check 1: each ProjectID should now appear exactly once
print("ProjectIDs unique:", df_clean["ProjectID"].is_unique)

# Rows removed = rows in the original data whose ObjectId is no longer in df_clean
dropped = df[~df["ObjectId"].isin(df_clean["ObjectId"])]
print("Rows removed:", len(dropped))

# Check 2: every removed row should be from a later year than the row kept for the same project
kept_year = df_clean.set_index("ProjectID")["InfraYear"]
print("All removed rows are later years:",
      (dropped["InfraYear"] > dropped["ProjectID"].map(kept_year)).all())

ProjectIDs unique: True
Rows removed: 28
All removed rows are later years: True


Result: 28 later-year rows were removed (9,855 → 9,827). Each `ProjectID` now
appears once, and every removed row was from a later year than the row kept.

### Drop columns with no useful information

Eight columns are removed because they hold no information for the analysis:
`Program` (empty), `infra_type`, `Creator`, `Editor`, `CreationDate`, and
`EditDate` (one identical value in every row), and `ObjectId` and `GlobalID`
(system row identifiers).

`ABC_String` and `ContractCost_String` are kept for now. Some rows contain text
such as "Clustered with Contract ID ...", which indicates projects that share a
contract and is reviewed in a later step.

In [12]:
cols_to_drop = ["Program", "infra_type", "Creator", "Editor",
                "CreationDate", "EditDate", "ObjectId", "GlobalID"]

df_clean = df_clean.drop(columns=cols_to_drop)

print(df_clean.shape)

(9827, 25)


In [13]:
# Confirm the dropped columns are gone and nothing else changed
print("Dropped columns still present:", [c for c in cols_to_drop if c in df_clean.columns])
print("Row count unchanged:", len(df_clean) == 9827)

Dropped columns still present: []
Row count unchanged: True


### Fix date columns

The three date columns are stored in different formats and are converted to
proper dates:

- `CompletionDateOriginal` is stored as milliseconds since 1970.
- `StartDate` is text in `MM/DD/YYYY` format. One value has a trailing space,
  so whitespace is stripped before converting.
- `CompletionDateActual` is text in `YYYY-MM-DD` format.

The 4 missing `CompletionDateOriginal` values are left empty, since the planned
date is unknown and should not be guessed. `CompletionYear` matches the year of
`CompletionDateActual` in every row, so it is used as is.

In [14]:
# Milliseconds since 1970 -> real date
df_clean["CompletionDateOriginal"] = pd.to_datetime(
    df_clean["CompletionDateOriginal"], unit="ms"
)

# Text like "03/24/2022"; strip the stray trailing space found in one value
df_clean["StartDate"] = pd.to_datetime(
    df_clean["StartDate"].str.strip(), format="%m/%d/%Y"
)

# Text like "2023-12-15"
df_clean["CompletionDateActual"] = pd.to_datetime(
    df_clean["CompletionDateActual"], format="%Y-%m-%d"
)

In [15]:
date_cols = ["StartDate", "CompletionDateOriginal", "CompletionDateActual"]

print(df_clean[date_cols].dtypes, "\n")
print("Missing values:\n", df_clean[date_cols].isna().sum(), "\n")
print("Earliest:\n", df_clean[date_cols].min(), "\n")
print("Latest:\n", df_clean[date_cols].max(), "\n")

# Logic check: no project should start after it was completed
print("Start after actual completion:",
      (df_clean["StartDate"] > df_clean["CompletionDateActual"]).sum())

StartDate                 datetime64[ns]
CompletionDateOriginal    datetime64[ns]
CompletionDateActual      datetime64[ns]
dtype: object 

Missing values:
 StartDate                 0
CompletionDateOriginal    4
CompletionDateActual      0
dtype: int64 

Earliest:
 StartDate                2018-02-26
CompletionDateOriginal   2018-10-11
CompletionDateActual     2022-07-01
dtype: datetime64[ns] 

Latest:
 StartDate                2025-03-14
CompletionDateOriginal   2026-02-04
CompletionDateActual     2025-05-30
dtype: datetime64[ns] 

Start after actual completion: 0


### Add project duration

A new column, `DurationDays`, measures the days from `StartDate` to
`CompletionDateActual`.

In [16]:
# Days from start to actual completion
df_clean["DurationDays"] = (
    df_clean["CompletionDateActual"] - df_clean["StartDate"]
).dt.days

In [17]:
print(df_clean.shape)
print(df_clean["DurationDays"].describe().round(0))
print("Projects over 1,500 days:", (df_clean["DurationDays"] > 1500).sum())
print("Non-positive durations:", (df_clean["DurationDays"] <= 0).sum())

(9827, 26)
count    9827.0
mean      247.0
std       135.0
min        14.0
25%       161.0
50%       228.0
75%       299.0
max      2012.0
Name: DurationDays, dtype: float64
Projects over 1,500 days: 7
Non-positive durations: 0


### Fill missing municipalities

638 projects have no `Municipality`. They are filled with "Unknown" so the rows
stay in the data and are visible as a group in charts and filters, instead of
being silently excluded. Other tidying of municipality names is skipped, since
municipality is not used in the analysis questions.

In [18]:
df_clean["Municipality"] = df_clean["Municipality"].fillna("Unknown")

## 4. Validate

Checking that the cleaned data matches the raw data apart from the intended
changes: the 28 later-year repeats removed and the columns dropped or added.

In [19]:
# Row count: raw rows minus the 28 later-year repeats
print("Rows: raw", len(df_raw), "-> clean", len(df_clean))

# Raw ContractCost total = clean total + cost of the removed repeat rows
removed_cost = (
    df_raw.sort_values(["ProjectID", "InfraYear"])
    .loc[lambda d: d.duplicated("ProjectID"), "ContractCost"]
    .sum()
)
print("Cost totals reconcile:",
      round(df_raw["ContractCost"].sum() - removed_cost - df_clean["ContractCost"].sum(), 2) == 0)

# Columns used in the analysis should have no missing values
key_cols = ["ProjectID", "Region", "Province", "Municipality", "Contractor",
            "TypeofWork", "ContractCost", "StartDate", "CompletionDateActual"]
print("Missing in key columns:", df_clean[key_cols].isna().sum().sum())

print("Unique ProjectIDs:", df_clean["ProjectID"].is_unique)
print("Shape:", df_clean.shape)

Rows: raw 9855 -> clean 9827
Cost totals reconcile: True
Missing in key columns: 0
Unique ProjectIDs: True
Shape: (9827, 26)


## 5. Export

The cleaned data is saved as an Excel file for the analysis notebook
(`02_eda.ipynb`) and the Power BI dashboard.

In [20]:
df_clean.to_excel("../data/flood_control_clean.xlsx", index=False, sheet_name="flood_control")

# Reload the saved file to confirm it was written correctly
saved = pd.read_excel("../data/flood_control_clean.xlsx")
print("Saved file shape:", saved.shape)

Saved file shape: (9827, 26)
